# Module 1 Pre-Fire Conditions
**Fire:** Cameron Peak, ignited 13 August 2020 · 208,913 acres · largest in Colorado history
**Landscape:** Cache la Poudre headwaters, Roosevelt National Forest, Larimer County CO

This repo is the foundation of the portfolio `soil-watershed-intelligence` and
`climate-resilience-indicators` both build on the toolkit demonstrated here.

Module 1 reconstructs the landscape as it stood before ignition: fuels, canopy
structure, vegetation moisture, terrain, and the antecedent weather. The point is
not description. It is to build a set of pre-fire predictors that Module 2 can
test against what actually burned.

**One methodological commitment that determines whether any of this is valid.**
Fuels come from LANDFIRE 2020 (the `200*` layer series), not the current
release. Current LANDFIRE already incorporates Cameron Peak as a disturbance, so
using it to describe pre-fire fuels and then correlating against severity is
circular so the fire is in the predictor. This is an easy mistake to make, it
produces impressively strong relationships, and those relationships are
artefacts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import daear_toolkit as dt
from daear_toolkit import data_access, indicators, viz
from daear_toolkit import fire_access as fa
from daear_toolkit import fire_indicators as fi
from daear_toolkit import climate_access as ca
from daear_toolkit import fire_access as fa

REGION = dt.POUDRE_CAMERON_PEAK
BBOX = REGION.bbox
IGNITION = pd.Timestamp("2020-08-13")

print(f"Region: {REGION.name}")
print(f"BBox:   {BBOX}")
print(f"Ignition: {IGNITION.date()}  |  Containment: 2020-12-02")

## Fuels and canopy structure

LANDFIRE via the Product Service an asynchronous job API, so the first call
submits a job and polls. Expect one to three minutes.

Four layers: Scott & Burgan 40 fuel models, canopy cover, canopy height, and
canopy base height. The last two are what separate a surface fire from a crown
fire, and they are absent from most quick-look fuel analyses.

In [ ]:
lf = fa.get_landfire(BBOX, layers=("fbfm40", "cc", "ch", "cbh", "cbd"), prefire=True)
print("LANDFIRE:", lf.attrs.get(""), "| layers:", list(lf.data_vars))

fuel_haz = fi.fuel_hazard(lf["fbfm40"])
crown = fi.crown_fire_potential(lf["cc"], lf["cbh"], lf["cbd"])

lf = fa.get_landfire(BBOX, layers=("fbfm40", "cc", "ch", "cbh", "cbd"), prefire=True)
print("LANDFIRE:", lf.attrs.get(""), "| layers:", list(lf.data_vars))

fuel_haz = fi.fuel_hazard(lf["fbfm40"])
crown = fi.crown_fire_potential(lf["cc"], lf["cbh"], lf["cbd"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
viz.plot_raster(lf["cc"], title="Canopy cover (%)", ax=axes[0], cmap="Greens")
viz.plot_raster(fuel_haz, title="Fuel model hazard (0-1)", ax=axes[1], cmap="YlOrRd")
viz.plot_raster(crown, title="Crown fire potential (screening)", ax=axes[2], cmap="YlOrRd")
plt.tight_layout()
plt.savefig("../outputs/01_fuels.png", dpi=150)
plt.show()

print(f"Area with crown fire potential > 0.6: {float((crown > 0.6).mean()):.1%}")
print("\nNote this is a screening index for canopy geometry only. No wind, no surface")
print("fire intensity, no foliar moisture. Where it is high, run a real fire behavior")
print("model; do not present it as a crown fire prediction.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
viz.plot_raster(lf["cc"], title="Canopy cover (%)", ax=axes[0], cmap="Greens")
viz.plot_raster(fuel_haz, title="Fuel model hazard (0-1)", ax=axes[1], cmap="YlOrRd")
viz.plot_raster(crown, title="Crown fire potential (screening)", ax=axes[2], cmap="YlOrRd")
plt.tight_layout()
plt.savefig("../outputs/01_fuels.png", dpi=150)
plt.show()

print(f"Area with crown fire potential > 0.6: {float((crown > 0.6).mean()):.1%}")
print("\nNote this is a screening index for canopy geometry only. No wind, no surface")
print("fire intensity, no foliar moisture. Where it is high, run a real fire behavior")
print("model; do not present it as a crown fire prediction.")

## Pre-fire vegetation moisture

NDMI (NIR vs SWIR1) rather than NDVI, because the question is dryness, not
greenness. Vegetation can stay green while its moisture content drops well below
the level that governs ignition and spread as NDVI cannot see that and NDMI can.

Three summers compared: 2018 and 2019 as reference, then the weeks immediately
before ignition in 2020.

In [ ]:
def summer_ndmi(year, start="07-15", end="08-10"):
    scene = data_access.get_optical_scene(BBOX, start=f"{year}-{start}", end=f"{year}-{end}", max_cloud_pct=15)
    return fi.ndmi(scene)

ndmi_by_year = {y: summer_ndmi(y) for y in (2018, 2019, 2020)}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, y in zip(axes[:3], (2018, 2019, 2020)):
    viz.plot_raster(ndmi_by_year[y], title=f"NDMI mid-July to mid-Aug {y}", ax=ax, cmap="BrBG", vmin=-0.2, vmax=0.5)

reference = (ndmi_by_year[2018] + ndmi_by_year[2019]) / 2
anomaly = ndmi_by_year[2020] - reference
viz.plot_raster(anomaly, title="2020 NDMI anomaly vs 2018-2019", ax=axes[3], cmap="RdBu", vmin=-0.2, vmax=0.2)
plt.tight_layout()
plt.savefig("../outputs/01_prefire_moisture.png", dpi=150)
plt.show()

print(f"Mean NDMI 2018-2019 reference: {float(reference.mean()):+.3f}")
print(f"Mean NDMI July-Aug 2020:       {float(ndmi_by_year[2020].mean()):+.3f}")
print(f"Anomaly:                       {float(anomaly.mean()):+.3f}")
print(f"Area drier than reference:     {float((anomaly < 0).mean()):.1%}")

## Antecedent fire weather

The 2020 Colorado fire season followed a dry winter and an exceptionally hot,
dry summer. gridMET ERC and VPD quantify how unusual conditions were in the
weeks before ignition.

Percentiles are computed against a **fixed 1991–2019 baseline**, deliberately
excluding 2020. Including the year being evaluated in its own reference
distribution drags the percentile toward the middle and understates how extreme
the conditions were.

In [ ]:
met = ca.get_gridmet(REGION, variables=("erc", "vpd", "pr", "tmmx"), start="1991-01-01", end="2020-12-31")
regional = met.mean(dim=[d for d in met.dims if d != "day"]).to_dataframe()

BASELINE = regional.loc["1991":"2019"]
window = regional.loc["2020-07-01":"2020-08-13"]

print("Conditions in the 6 weeks before ignition, vs the 1991-2019 baseline for the same calendar window:\n")
for var, label in [("erc", "Energy Release Component"), ("vpd", "Vapor Pressure Deficit (kPa)"),
                   ("tmmx", "Max temperature (C)")]:
    base_window = BASELINE[(BASELINE.index.month.isin([7, 8])) & (BASELINE.index.day <= 13)][var]
    pct = (base_window < window[var].mean()).mean() * 100
    print(f"  {label:<32} 2020 mean {window[var].mean():7.2f}   baseline mean {base_window.mean():7.2f}   "
          f"-> {pct:.0f}th percentile")

water_year_precip = regional["pr"].loc["2019-10-01":"2020-08-13"].sum()
base_wy = [BASELINE["pr"].loc[f"{y-1}-10-01":f"{y}-08-13"].sum() for y in range(1992, 2020)]
print(f"\n  Oct-Aug precipitation, WY2020:   {water_year_precip:.0f} mm")
print(f"  Baseline mean:                   {np.mean(base_wy):.0f} mm")
print(f"  -> {(np.array(base_wy) < water_year_precip).mean()*100:.0f}th percentile")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, var, label in zip(axes, ["erc", "vpd"], ["ERC", "VPD (kPa)"]):
    clim = BASELINE.groupby([BASELINE.index.month, BASELINE.index.day])[var].agg(["mean", "std"])
    doy_2020 = regional.loc["2020"]
    idx = pd.MultiIndex.from_arrays([doy_2020.index.month, doy_2020.index.day])
    ax.fill_between(doy_2020.index, clim.loc[idx, "mean"].values - clim.loc[idx, "std"].values,
                    clim.loc[idx, "mean"].values + clim.loc[idx, "std"].values,
                    color="grey", alpha=0.25, label="1991-2019 mean +/- 1 sd")
    ax.plot(doy_2020.index, clim.loc[idx, "mean"].values, color="grey", lw=1)
    ax.plot(doy_2020.index, doy_2020[var].values, color="#b5443a", lw=1.4, label="2020")
    ax.axvline(IGNITION, color="black", ls="--", lw=1.2, label="ignition")
    ax.set_ylabel(label); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/01_fire_weather_2020.png", dpi=150)
plt.show()

## Composite pre-fire condition index

Fuels, dryness, crown structure, and terrain combined into one screening layer.

Weights are stated and they are testable. The weight-sensitivity machinery in 
`climate_indicators` works on any indicator table, so the same Monte Carlo used 
in `climate-resilience-indicators` Module 3 applies directly here if this index 
ever needs defending.

**This index is not a fire prediction.** It describes where the landscape was
predisposed to burn severely *if* fire arrived. Whether fire arrived is a
function of ignition and weather, which no landscape layer contains.

In [ ]:
terrain = data_access.get_terrain(BBOX)

# NDMI is a moisture index, so invert it to get a deficit that points the same
# direction as the other components: higher = worse.
moisture_deficit = -ndmi_by_year[2020]

pci = fi.prefire_condition_index(
    fuel_hazard_layer=fuel_haz,
    moisture_deficit=moisture_deficit,
    crown_potential=crown,
    slope_deg=terrain["slope_deg"],
    weights=(0.30, 0.30, 0.25, 0.15),
)

fig, ax = plt.subplots(figsize=(8, 6.5))
viz.plot_raster(pci, title="Pre-fire condition index (Aug 2020)", ax=ax, cmap="YlOrRd")
plt.tight_layout()
plt.savefig("../outputs/01_prefire_condition_index.png", dpi=150)
plt.show()

pci.to_netcdf("../outputs/01_prefire_condition_index.nc")
for layer, name in [(fuel_haz, "fuel_hazard"), (crown, "crown_potential"), (pci, "composite")]:
    print(f"{name:<18} mean={float(layer.mean()):.3f}  >0.7: {float((layer > 0.7).mean()):.1%}")

## Summary

The pre-fire landscape: LANDFIRE 2020 fuels and canopy structure, Sentinel-2
moisture with a two-year reference, gridMET fire weather in percentile terms,
and a composite condition index.

`01_prefire_condition_index.nc` is saved for Module 2, which tests whether any of
it predicted where the fire actually burned severely.

**Carry these forward:**

- LANDFIRE 2020, not current. Using post-fire LANDFIRE would make Module 2
  circular. This is the single most important decision in this notebook.
- The crown-fire layer is canopy geometry only as a screening index, not a fire
  behavior model.
- Fuel hazard collapses categorical fuel models onto an ordinal scale so they can
  enter a composite. Fuel models are not truly ordered; this is a stated
  simplification.
- The composite describes predisposition given ignition. It says nothing about
  ignition probability, which is where most of the variance in whether a place
  burns actually lives.